# SciVer Full-Search v3 server notebook

Canonical thin operator notebook for a repository that was prepared manually before JupyterLab starts. Run the ordered stages below. The default confirmation at every live boundary is empty, so it makes no HTTP, solver, or proposer call. Never save outputs after entering runtime credentials.

## 1. SETUP — locate and validate the existing repository

In [ ]:
import importlib
import os
import sys
from getpass import getpass
from pathlib import Path

def _locate_repository_root() -> Path:
    candidates = (Path.cwd(), *Path.cwd().parents)
    for candidate in candidates:
        if (candidate / '.git').exists() and (candidate / 'notebooks' / 'sciver_full_search_v3_server.ipynb').is_file():
            return candidate.resolve()
    raise RuntimeError('Open JupyterLab from inside the manually prepared repository checkout')

def _required_setting(name: str, prompt: str) -> str:
    value = os.environ.get(name) or input(prompt)
    if not value.strip():
        raise RuntimeError(f'{name} is required')
    return value.strip()

REPOSITORY_DIRECTORY = _locate_repository_root()
PINNED_COMMIT_SHA = _required_setting('PINNED_COMMIT_SHA', 'Pinned 40-character commit SHA: ')
DATASET_PATH = Path(_required_setting('SCIVER_DATASET_PATH', 'Absolute SciVer testset path: ')).expanduser().resolve()
RUN_ID = _required_setting('SCIVER_RUN_ID', 'Run ID: ')
PREPARATION_DIRECTORY = REPOSITORY_DIRECTORY / 'workspace' / 'meta_harness' / 'full_search_v3' / RUN_ID / 'preparation'
CONFIG_PATH = None

if str(REPOSITORY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIRECTORY))

from meta_harness.full_search_v3_server import (
    EXPECTED_REPOSITORY_ORIGIN,
    FullSearchV3ServerError,
    freeze_full_search_v3_server_winner,
    inspect_full_search_v3_server_final_status,
    inspect_full_search_v3_server_smoke_receipt,
    inspect_full_search_v3_server_status,
    preflight_full_search_v3_server_final,
    preflight_full_search_v3_server_run,
    prepare_full_search_v3_server_run,
    run_full_search_v3_server_smoke,
    start_or_resume_full_search_v3_server_final,
    start_or_resume_full_search_v3_server_run,
    validate_full_search_v3_server_checkout,
)

checkout = validate_full_search_v3_server_checkout(
    repository_root=REPOSITORY_DIRECTORY,
    pinned_commit_sha=PINNED_COMMIT_SHA,
    expected_origin_url=EXPECTED_REPOSITORY_ORIGIN,
)
required_imports = ('requests', 'PIL', 'meta_harness.full_search_v3_server')
for module_name in required_imports:
    importlib.import_module(module_name)
if not DATASET_PATH.is_file():
    raise FileNotFoundError('Configured SciVer dataset path does not exist')
{'checkout': checkout, 'required_imports': required_imports, 'dataset_present': True}

Environment creation and `python -m pip install -r requirements.txt` happen before JupyterLab is launched. This notebook verifies dependency metadata and imports; it does not install packages or modify the checkout.

## 2. SETUP — prepare or verify the deterministic split

In [ ]:
prepared = prepare_full_search_v3_server_run(
    dataset_path=DATASET_PATH,
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    preparation_directory=PREPARATION_DIRECTORY,
    config_path=CONFIG_PATH,
)
{
    'run_id': prepared['run_id'],
    'split_sha256': prepared['split_sha256'],
    'search': prepared['SEARCH'],
    'final': prepared['FINAL'],
    'sample_overlap_count': prepared['sample_overlap_count'],
    'paper_overlap_count': prepared['paper_overlap_count'],
}

## 3. OFFLINE_SMOKE — repository-owned request and run preflight

In [ ]:
offline_smoke = preflight_full_search_v3_server_run(
    repository_root=REPOSITORY_DIRECTORY,
    run_id=RUN_ID,
    search_safe_manifest_path=prepared['search_safe_manifest_path'],
    search_records_path=prepared['search_dataset_path'],
    source_commit=PINNED_COMMIT_SHA,
)
{
    key: offline_smoke[key]
    for key in ('protocol_id', 'run_id', 'resume', 'resume_identity_checked', 'config_sha256', 'split_sha256', 'search_membership_sha256', 'canonical_p0_prompt_sha256', 'solver', 'parser_version', 'offline_smoke', 'workload', 'checkpoints')
}

## 4. LIVE_SMOKE — authorized model-list preflight and one canonical P0 POST

In [ ]:
def _runtime_api_values() -> tuple[str, str]:
    raw_url = os.environ.get('API_URL')
    if raw_url is None:
        raw_url = input('API_URL base (runtime only): ')
    raw_key = os.environ.get('API_KEY')
    if raw_key is None:
        raw_key = getpass('API_KEY (runtime only): ')
    if '\r' in raw_key or '\n' in raw_key:
        raise RuntimeError('API_KEY contains invalid characters')
    api_url = raw_url.strip()
    api_key = raw_key.strip()
    if not api_url or not api_key:
        raise RuntimeError('API_URL and API_KEY are required for an authorized live stage')
    return api_url, api_key

LIVE_SMOKE_CONFIRMATION = input('Type RUN_LIVE_SMOKE to authorize live smoke, or press Enter to skip: ').strip()
if LIVE_SMOKE_CONFIRMATION == 'RUN_LIVE_SMOKE':
    runtime_api_url, runtime_api_key = _runtime_api_values()
    try:
        live_smoke = run_full_search_v3_server_smoke(
            repository_root=REPOSITORY_DIRECTORY,
            run_id=RUN_ID,
            search_safe_manifest_path=prepared['search_safe_manifest_path'],
            search_records_path=prepared['search_dataset_path'],
            authorize_smoke_execution=True,
            api_url=runtime_api_url,
            api_key=runtime_api_key,
            source_commit=PINNED_COMMIT_SHA,
        )
    finally:
        runtime_api_url = runtime_api_key = None
    live_smoke
elif LIVE_SMOKE_CONFIRMATION:
    raise RuntimeError('LIVE_SMOKE requires the exact confirmation RUN_LIVE_SMOKE')
else:
    print('LIVE_SMOKE skipped; no credential was read and no HTTP request was made.')

## 5. Smoke receipt validation

In [ ]:
try:
    smoke_receipt = inspect_full_search_v3_server_smoke_receipt(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
except FullSearchV3ServerError:
    smoke_receipt = {'status': 'missing_or_incompatible'}
smoke_receipt

## 6. FULL_SEARCH — separately authorized start or resume

In [ ]:
FULL_SEARCH_CONFIRMATION = input('Type RUN_FULL_SEARCH to authorize SEARCH, or press Enter to skip: ').strip()
if FULL_SEARCH_CONFIRMATION == 'RUN_FULL_SEARCH':
    runtime_api_url, runtime_api_key = _runtime_api_values()
    try:
        search_result = start_or_resume_full_search_v3_server_run(
            repository_root=REPOSITORY_DIRECTORY,
            run_id=RUN_ID,
            search_safe_manifest_path=prepared['search_safe_manifest_path'],
            search_records_path=prepared['search_dataset_path'],
            authorize_search_execution=True,
            api_url=runtime_api_url,
            api_key=runtime_api_key,
            source_commit=PINNED_COMMIT_SHA,
        )
    finally:
        runtime_api_url = runtime_api_key = None
    search_result['search']
elif FULL_SEARCH_CONFIRMATION:
    raise RuntimeError('FULL_SEARCH requires the exact confirmation RUN_FULL_SEARCH')
else:
    print('FULL_SEARCH skipped; no credential, solver, or proposer was accessed.')

## 7. SEARCH status

In [ ]:
try:
    search_status = inspect_full_search_v3_server_status(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
except FullSearchV3ServerError:
    search_status = {'status': 'not_started'}
search_status

## 8. Freeze — immutable SEARCH-only winner

In [ ]:
frozen_winner = None
if search_status.get('status') in {'patience_stopped', 'max_stopped'}:
    frozen_winner = freeze_full_search_v3_server_winner(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
    frozen_winner
else:
    print('SEARCH is not terminal; no winner was frozen and FINAL remains locked.')

## 9. FINAL preflight — offline and frozen-identity bound

In [ ]:
if frozen_winner is None:
    final_preflight = {'status': 'locked_until_search_freeze'}
else:
    final_preflight = preflight_full_search_v3_server_final(
        repository_root=REPOSITORY_DIRECTORY,
        run_id=RUN_ID,
        dataset_path=DATASET_PATH,
        private_manifest_path=prepared['private_manifest_path'],
        search_safe_manifest_path=prepared['search_safe_manifest_path'],
    )
final_preflight

## 10. FINAL — separately authorized paired P0/P* execution

In [ ]:
FINAL_CONFIRMATION = input('Type RUN_FINAL_ONCE to authorize paired FINAL, or press Enter to skip: ').strip()
if FINAL_CONFIRMATION == 'RUN_FINAL_ONCE':
    if frozen_winner is None:
        raise RuntimeError('FINAL remains locked until a valid winner is frozen')
    runtime_api_url, runtime_api_key = _runtime_api_values()
    try:
        final_result = start_or_resume_full_search_v3_server_final(
            repository_root=REPOSITORY_DIRECTORY,
            run_id=RUN_ID,
            dataset_path=DATASET_PATH,
            private_manifest_path=prepared['private_manifest_path'],
            search_safe_manifest_path=prepared['search_safe_manifest_path'],
            authorize_final_execution=True,
            api_url=runtime_api_url,
            api_key=runtime_api_key,
        )
    finally:
        runtime_api_url = runtime_api_key = None
    final_result['final']
elif FINAL_CONFIRMATION:
    raise RuntimeError('FINAL requires the exact confirmation RUN_FINAL_ONCE')
else:
    print('FINAL skipped; no credential was read and no FINAL request was made.')

## 11. Sanitized aggregate reporting

In [ ]:
try:
    final_status = inspect_full_search_v3_server_final_status(
        repository_root=REPOSITORY_DIRECTORY, run_id=RUN_ID
    )
except FullSearchV3ServerError:
    final_status = {'status': 'not_started'}
{
    'run_id': RUN_ID,
    'search': search_status,
    'frozen_winner': frozen_winner,
    'final': final_status,
    'artifact_root': str(REPOSITORY_DIRECTORY / 'workspace' / 'meta_harness' / 'full_search_v3' / RUN_ID),
}